# Fusions CV (2-way + 3-way over the 9 base models)

Same protocol as `base_models_cv.ipynb` — candidate-level splits, MIN_SPEAKING_S=30, both rotations A and B — with a fusion overlay on top.

## Why this gives correct + reliable numbers (no leakage)
Per (rotation, fold):
1. Fit each of the 9 base models on the **train** side only.
2. Score the **CV** side and the **test** side with each base model. Cache `(p_cv, p_te)` per model.
3. For every 2-way pair and every 3-way triple of base models, **search fusion weights on the CV side only**, separately for each threshold-strategy:
    - 2-way: α ∈ {0.00, 0.05, …, 1.00} (21 points)
    - 3-way: simplex grid step 0.1 → 66 points
    - For strategy `F1`: pick (weights, thr) that **maximise CV F1**.
    - For strategy `P{X}`: pick (weights, thr) that **maximise CV recall** under the constraint `CV precision ≥ X%` (and ≥3 true positives).
4. Apply the frozen `(weights, thr)` to the cached test-side probas. Report `te_*` metrics + gap.

Test-side data is never used for weight selection or threshold selection. Per-strategy weight search means the P95 numbers come from weights *optimised for P95*, not weights optimised for F1 then evaluated at P95 — that's the difference between this notebook and a typical "fusion-by-F1" pipeline.

## What's in vs out
- **In**: the same 9 base models from `base_models_cv.ipynb` (5 text-XGB, whisper RF/XGB, wavlm RF/XGB).
- **In**: 2-way pairs — C(9,2) = 36 candidates; 3-way triples — C(9,3) = 84 candidates.
- **Out**: stacking, meta-LR, isotonic calibration, 4-way+ fusions. (Each can go in a follow-up if 2-way/3-way numbers don't move the needle.)

## Honest-numbers caveat (worth knowing, not worth panicking about)
Both fusion **weights** and the **threshold** are picked on the same CV pool. That double-use is the standard fusion pattern but can mildly inflate CV numbers. The size of any inflation shows up as a positive `gap_primary` (cv − te) — that's exactly the column you should look at. A small symmetric gap on both rotations = trustworthy; a large or asymmetric gap = the search overfit the CV pool.

## Outputs
- `checkpoints_fusions/per_fold.csv` — every rotation × fold × candidate × strategy row
- `checkpoints_fusions/summary_avg.csv` — averaged across folds
- Display: per (rotation, strategy) — top-1 base + top-5 2-way + top-5 3-way ranked by avg CV primary metric, with `delta_vs_best_base` so the fusion lift is visible at a glance.

In [ ]:
# === 0. Setup ===
from pathlib import Path
import re, warnings, time, itertools
import numpy as np
import pandas as pd

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

warnings.filterwarnings('ignore')

NB_DIR        = Path('.').resolve()
SAVE_DIR      = NB_DIR / 'checkpoints_fusions'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
DURATIONS_DIR = NB_DIR / 'checkpoints_honest_eval'

# Split protocol — must match base_models_cv.ipynb to keep numbers comparable
MIN_SPEAKING_S  = 30
N_FOLDS         = 5
CV_FRAC_ALWAYS  = 0.20
CV_FRAC_TEST    = 0.40
RANDOM_SEED     = 42

ROTATIONS = [
    {'name': 'A', 'always': 'audios4', 'test': 'audios5'},
    {'name': 'B', 'always': 'audios5', 'test': 'audios4'},
]

DEPLOY_POS_RATE = 0.17
SPW_DEPLOY      = (1.0 - DEPLOY_POS_RATE) / DEPLOY_POS_RATE

# Threshold-pick strategies
STRATEGIES = ['F1','P80','P85','P90','P95']
PREC_FLOOR = {'P80':0.80,'P85':0.85,'P90':0.90,'P95':0.95}

# Fusion search grids
ALPHA_STEP   = 0.05    # 2-way alpha step (21 points)
SIMPLEX_STEP = 0.10    # 3-way simplex step (66 points)

# Display
TOP_2WAY = 5
TOP_3WAY = 5

# Vectorised threshold grids
F1_THR_GRID = np.arange(0.20, 0.81, 0.01)
P_THR_GRID  = np.arange(0.99, 0.10, -0.01)

LABEL_MAP = {
    'read':1,'cheating':1,'reading':1,'scripted':1,'yes':1,'1':1,1:1,
    'spontaneous':0,'not cheating':0,'not_cheating':0,'no':0,'0':0,0:0,'genuine':0,
}

ALL_TEXT_FEATURES = [
    'filler_rate','filler_count','repetition_rate','repair_rate','discourse_marker_rate','hedge_rate',
    'ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
    'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
    'noun_rate','verb_rate','adj_rate',
    'pause_mean','pause_std','pause_median','pause_skew','long_pause_rate','pause_ratio','n_pauses',
    'pause_regularity','pause_before_content_ratio','pause_before_function_ratio','mid_phrase_pause_rate',
    'words_per_sec','articulation_rate','initial_pause','longest_pause',
    'suspicious_gap_count','suspicious_gap_ratio',
    'formal_transition_count','formal_transition_rate','ai_phrase_count','ai_phrase_rate',
    'f0_mean','f0_std','f0_range','f0_skew','f0_slope','energy_mean','energy_std','speaking_rate_std',
    'jitter_local','shimmer_local','hnr_mean',
    'mean_perplexity','burstiness',
]
STYLO_FEATS = ['ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
               'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
               'noun_rate','verb_rate','adj_rate']
TEXT_RANK_COLS = [f for f in ALL_TEXT_FEATURES
                  if f not in ('f0_mean','f0_std','f0_range','f0_skew','f0_slope',
                               'energy_mean','energy_std','speaking_rate_std',
                               'jitter_local','shimmer_local','hnr_mean',
                               'mean_perplexity','burstiness')]

BATCHES = ['audios2','audios4','audios5']
print(f'NB_DIR={NB_DIR}\nSAVE_DIR={SAVE_DIR}')
print(f'MIN_SPEAKING_S={MIN_SPEAKING_S}  N_FOLDS={N_FOLDS}  '
      f'CV_FRAC_ALWAYS={CV_FRAC_ALWAYS}  CV_FRAC_TEST={CV_FRAC_TEST}')
print(f'SPW_DEPLOY={SPW_DEPLOY:.2f}  ALPHA_STEP={ALPHA_STEP}  SIMPLEX_STEP={SIMPLEX_STEP}')
print(f'ROTATIONS: ' + ', '.join(f'{r["name"]}=(always={r["always"]}, test={r["test"]})' for r in ROTATIONS))

In [ ]:
# === 1. Data loading + duration filter + candidate_id (same as base_models_cv) ===
WAVLM_WHOLE_CANDIDATES = lambda n: [f'{n}_wavlm_whole.csv', f'{n}_whole_pretrained.csv']

def _first_existing(cands):
    for c in cands:
        p = NB_DIR / c
        if p.exists(): return p
    raise FileNotFoundError(f'None of {cands} exist under {NB_DIR}')

def load_gt(name):
    gt = pd.read_csv(NB_DIR / f'{name}GT.csv')
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth'))
    gt = gt.rename(columns={fn_col:'filename', lbl_col:'label_raw'})
    gt['label_int'] = gt['label_raw'].map(
        lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    return gt[gt['label_int'].isin([0,1])][['filename','label_int']]

def load_durations(name):
    p = DURATIONS_DIR / f'{name}_durations.csv'
    if not p.exists():
        print(f'  WARN: {p} missing — duration filter will be a no-op for {name}')
        return None
    return pd.read_csv(p)[['filename','speaking_time_s']]

def load_folder(name):
    gt   = load_gt(name)
    text = pd.read_csv(NB_DIR / f'{name}_features.csv')
    df   = gt.merge(text, on='filename', how='inner')
    wp   = pd.read_csv(_first_existing(WAVLM_WHOLE_CANDIDATES(name)))
    df   = df.merge(wp, on='filename', how='inner')
    wh   = pd.read_csv(NB_DIR / f'{name}_whisper_whole.csv')
    df   = df.merge(wh, on='filename', how='inner')
    df['batch'] = name
    dur = load_durations(name)
    if dur is not None:
        df = df.merge(dur, on='filename', how='left')
    return df

def filter_by_duration(df, min_s):
    if min_s <= 0 or 'speaking_time_s' not in df.columns: return df
    keep = (df['speaking_time_s'] >= min_s) | df['speaking_time_s'].isna()
    return df[keep].reset_index(drop=True)

_RE_CAND = re.compile(r'^(.+)_(\d{1,3})\.[a-zA-Z0-9]+$')
def attach_candidate_id(df):
    df = df.copy()
    df['candidate_id'] = df['filename'].astype(str).map(
        lambda f: (_RE_CAND.match(f).group(1) if _RE_CAND.match(f) else None))
    return df

batches_full = {b: attach_candidate_id(load_folder(b))            for b in BATCHES}
batches      = {b: filter_by_duration(batches_full[b], MIN_SPEAKING_S) for b in BATCHES}

print('=== Per-batch counts (raw -> filtered at >= {}s) ==='.format(MIN_SPEAKING_S))
for b in BATCHES:
    f0, f1 = batches_full[b], batches[b]
    y0, y1 = f0['label_int'].values, f1['label_int'].values
    print(f'  {b}:  rows {len(f0):4d} -> {len(f1):4d}   '
          f'cheat {int((y0==1).sum()):3d}->{int((y1==1).sum()):3d}   '
          f'honest {int((y0==0).sum()):3d}->{int((y1==0).sum()):3d}   '
          f'candidates {f0["candidate_id"].nunique()}->{f1["candidate_id"].nunique()}')

In [ ]:
# === 2. Feature columns + top-N text feature ranking on audios2 only ===
first = batches['audios2']
WH_COLS    = [c for c in first.columns if c.startswith('whisper_')]
WP_COLS    = [c for c in first.columns if c.startswith('wavlm_')
              and not c.startswith('wavlm_mean_') and not c.startswith('wavlm_std_')]
TEXT_ALL   = [c for c in ALL_TEXT_FEATURES if c in first.columns]
TEXT_STYLO = [c for c in STYLO_FEATS       if c in first.columns]
TEXT_RANK  = [c for c in TEXT_RANK_COLS    if c in first.columns]

_X = first[TEXT_RANK].fillna(0).values
_y = first['label_int'].values
_sc = StandardScaler().fit(_X)
_rkr = xgb.XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
    random_state=RANDOM_SEED)
_rkr.fit(_sc.transform(_X), _y)
_imp = pd.Series(_rkr.feature_importances_, index=TEXT_RANK).sort_values(ascending=False)
TEXT_TOP10 = _imp.head(10).index.tolist()
TEXT_TOP15 = _imp.head(15).index.tolist()
TEXT_TOP20 = _imp.head(20).index.tolist()

print(f'  whisper_wp d={len(WH_COLS)}  wavlm_wp d={len(WP_COLS)}  '
      f'text_all d={len(TEXT_ALL)}  text_stylo d={len(TEXT_STYLO)}  text_rank d={len(TEXT_RANK)}')

In [ ]:
# === 3. Model factories + registry (same 9 as base_models_cv) ===
def make_xgb(n_feats, seed=RANDOM_SEED):
    cs = 0.3 if n_feats > 500 else 0.8
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=cs, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
        random_state=seed)

def make_rf(n_feats, seed=RANDOM_SEED):
    return RandomForestClassifier(
        n_estimators=500, max_depth=8, min_samples_leaf=3,
        class_weight={0:1.0, 1:float(SPW_DEPLOY)},
        n_jobs=-1, random_state=seed)

def mk_X(cols): return lambda d: d[cols].fillna(0).values

BASE_REGISTRY = {
    'text_top10_xgb':  (mk_X(TEXT_TOP10), lambda s=RANDOM_SEED: make_xgb(len(TEXT_TOP10), s)),
    'text_top15_xgb':  (mk_X(TEXT_TOP15), lambda s=RANDOM_SEED: make_xgb(len(TEXT_TOP15), s)),
    'text_top20_xgb':  (mk_X(TEXT_TOP20), lambda s=RANDOM_SEED: make_xgb(len(TEXT_TOP20), s)),
    'text_stylo_xgb':  (mk_X(TEXT_STYLO), lambda s=RANDOM_SEED: make_xgb(len(TEXT_STYLO), s)),
    'text_all_xgb':    (mk_X(TEXT_ALL),   lambda s=RANDOM_SEED: make_xgb(len(TEXT_ALL),   s)),
    'whisper_wp_rf':   (mk_X(WH_COLS),    lambda s=RANDOM_SEED: make_rf (len(WH_COLS),    s)),
    'whisper_wp_xgb':  (mk_X(WH_COLS),    lambda s=RANDOM_SEED: make_xgb(len(WH_COLS),    s)),
    'wavlm_wp_rf':     (mk_X(WP_COLS),    lambda s=RANDOM_SEED: make_rf (len(WP_COLS),    s)),
    'wavlm_wp_xgb':    (mk_X(WP_COLS),    lambda s=RANDOM_SEED: make_xgb(len(WP_COLS),    s)),
}
MODELS = list(BASE_REGISTRY.keys())

# Short aliases for compact display
ALIAS = {
    'text_top10_xgb':'t10', 'text_top15_xgb':'t15', 'text_top20_xgb':'t20',
    'text_stylo_xgb':'tst', 'text_all_xgb':'tal',
    'whisper_wp_rf':'wh.rf', 'whisper_wp_xgb':'wh.xg',
    'wavlm_wp_rf':'wv.rf',   'wavlm_wp_xgb':'wv.xg',
}
print('Models:', MODELS)
print('Aliases:', ALIAS)
print(f'2-way candidates: C(9,2) = {len(list(itertools.combinations(MODELS,2)))}')
print(f'3-way candidates: C(9,3) = {len(list(itertools.combinations(MODELS,3)))}')

In [ ]:
# === 4. Candidate-level fold split (same as base_models_cv) ===
def candidate_split(df, frac_holdout, seed):
    rng = np.random.default_rng(seed)
    cand_label = df.groupby('candidate_id')['label_int'].max()
    pos_cands = cand_label[cand_label==1].index.tolist()
    neg_cands = cand_label[cand_label==0].index.tolist()
    rng.shuffle(pos_cands); rng.shuffle(neg_cands)
    n_pos_h = max(1, int(round(len(pos_cands) * frac_holdout)))
    n_neg_h = max(1, int(round(len(neg_cands) * frac_holdout)))
    holdout = set(pos_cands[:n_pos_h] + neg_cands[:n_neg_h])
    train_part   = df[~df['candidate_id'].isin(holdout)].reset_index(drop=True)
    holdout_part = df[ df['candidate_id'].isin(holdout)].reset_index(drop=True)
    return train_part, holdout_part

def build_fold(rotation, fold_idx):
    seed = RANDOM_SEED + fold_idx
    always_train, always_cv = candidate_split(batches[rotation['always']], CV_FRAC_ALWAYS, seed=seed)
    test_test,    test_cv   = candidate_split(batches[rotation['test']],   CV_FRAC_TEST,   seed=seed + 1000)
    df_train = pd.concat([batches['audios2'], always_train], ignore_index=True)
    df_cv    = pd.concat([always_cv, test_cv],                ignore_index=True)
    df_test  = test_test.reset_index(drop=True)
    cv_cands    = set(df_cv['candidate_id'])
    test_cands  = set(df_test['candidate_id'])
    train_cands = set(df_train['candidate_id'])
    assert not (cv_cands & test_cands),    f'rot {rotation["name"]} fold {fold_idx}: CV/test candidate overlap'
    assert not (train_cands & test_cands), f'rot {rotation["name"]} fold {fold_idx}: train/test candidate overlap'
    return df_train, df_cv, df_test

for rot in ROTATIONS:
    _tr, _cv, _te = build_fold(rot, 0)
    y_tr, y_cv_, y_te_ = _tr['label_int'].values, _cv['label_int'].values, _te['label_int'].values
    print(f'rot {rot["name"]} fold 0:  '
          f'tr n={len(_tr)} (+{int((y_tr==1).sum())})  '
          f'cv n={len(_cv)} (+{int((y_cv_==1).sum())})  '
          f'te n={len(_te)} (+{int((y_te_==1).sum())})')

In [ ]:
# === 5. Vectorised threshold pickers (the speed trick that makes 3-way search tractable) ===
def _best_f1_thr_v(p, y):
    """Vectorised: scan F1_THR_GRID at once, return (thr, f1) at the argmax."""
    if len(p) == 0:
        return 0.5, 0.0
    y_b   = y.astype(bool)
    preds = (p[None, :] >= F1_THR_GRID[:, None])
    tp = (preds & y_b[None, :]).sum(axis=1)
    fp = (preds & ~y_b[None, :]).sum(axis=1)
    fn = (~preds & y_b[None, :]).sum(axis=1)
    denom = (2*tp + fp + fn)
    f1 = np.where(denom > 0, 2*tp / np.maximum(denom, 1), 0.0)
    k = int(np.argmax(f1))
    return float(F1_THR_GRID[k]), float(f1[k])

def _best_rec_at_prec_v(p, y, target, min_tp=3):
    """Vectorised: returns (thr, recall, precision) maximising recall under prec >= target."""
    if len(p) == 0:
        return None, None, None
    y_b   = y.astype(bool)
    preds = (p[None, :] >= P_THR_GRID[:, None])
    tp = (preds & y_b[None, :]).sum(axis=1)
    fp = (preds & ~y_b[None, :]).sum(axis=1)
    fn = (~preds & y_b[None, :]).sum(axis=1)
    prec = np.where((tp+fp) > 0, tp / np.maximum(tp+fp, 1), 0.0)
    rec  = np.where((tp+fn) > 0, tp / np.maximum(tp+fn, 1), 0.0)
    feas = (tp >= min_tp) & (prec >= target)
    if not feas.any():
        return None, None, None
    rec_eff = np.where(feas, rec, -1.0)
    k = int(np.argmax(rec_eff))
    return float(P_THR_GRID[k]), float(rec[k]), float(prec[k])

def _metrics_at(p, y, thr):
    if thr is None or len(p)==0:
        return dict(prec=np.nan, rec=np.nan, f1=np.nan)
    pred = (p >= thr).astype(int)
    return dict(
        prec=float(precision_score(y, pred, zero_division=0)),
        rec =float(recall_score(y, pred, zero_division=0)),
        f1  =float(f1_score(y, pred, zero_division=0)),
    )

def fit_score(name, df_tr, df_cv, df_te, seed=RANDOM_SEED):
    X_fn, factory = BASE_REGISTRY[name]
    Xtr, ytr = X_fn(df_tr), df_tr['label_int'].values
    sc = StandardScaler().fit(Xtr)
    clf = factory(seed)
    clf.fit(sc.transform(Xtr), ytr)
    p_cv = clf.predict_proba(sc.transform(X_fn(df_cv)))[:, 1]
    p_te = clf.predict_proba(sc.transform(X_fn(df_te)))[:, 1]
    return p_cv, df_cv['label_int'].values, p_te, df_te['label_int'].values

print('Vectorised pickers + fit_score ready.')

In [ ]:
# === 6. Fusion weight search (per strategy, on CV side only) ===
ALPHAS_2WAY = np.arange(0.0, 1.0 + 1e-9, ALPHA_STEP)
_W3_GRID = []
for _w1 in np.arange(0.0, 1.0 + 1e-9, SIMPLEX_STEP):
    for _w2 in np.arange(0.0, 1.0 - _w1 + 1e-9, SIMPLEX_STEP):
        _w3 = max(0.0, 1.0 - _w1 - _w2)
        _W3_GRID.append((round(float(_w1),2), round(float(_w2),2), round(float(_w3),2)))
_W3_GRID = np.array(_W3_GRID)
print(f'2-way alpha grid: {len(ALPHAS_2WAY)} points')
print(f'3-way simplex grid: {len(_W3_GRID)} points')

def _strategy_score(p_cv, y_cv, strategy):
    """Return (thr, primary, info_dict) for a given fused CV vector under one strategy.
    primary = F1 for 'F1', recall for 'P{X}'. Returns (None, None, None) if infeasible."""
    if strategy == 'F1':
        thr, f1 = _best_f1_thr_v(p_cv, y_cv)
        return thr, f1, {'cv_prec': None, 'cv_rec': None, 'cv_f1': f1}
    thr, rec, prec = _best_rec_at_prec_v(p_cv, y_cv, PREC_FLOOR[strategy])
    if thr is None:
        return None, None, None
    return thr, rec, {'cv_prec': prec, 'cv_rec': rec, 'cv_f1': None}

def search_2way(pa_cv, pb_cv, y_cv, strategy):
    """Returns (alpha, thr, primary) maximising the strategy's primary metric on CV.
    None if no alpha is feasible (only happens for P-strategies)."""
    best = None
    for alpha in ALPHAS_2WAY:
        fused = alpha*pa_cv + (1-alpha)*pb_cv
        thr, primary, _ = _strategy_score(fused, y_cv, strategy)
        if thr is None: continue
        if best is None or primary > best[2]:
            best = (round(float(alpha),2), float(thr), float(primary))
    return best

def search_3way(pa_cv, pb_cv, pc_cv, y_cv, strategy):
    """Returns ((w1,w2,w3), thr, primary). None if infeasible (P-strategies)."""
    best = None
    for w in _W3_GRID:
        fused = w[0]*pa_cv + w[1]*pb_cv + w[2]*pc_cv
        thr, primary, _ = _strategy_score(fused, y_cv, strategy)
        if thr is None: continue
        if best is None or primary > best[2]:
            best = ((round(float(w[0]),2), round(float(w[1]),2), round(float(w[2]),2)),
                    float(thr), float(primary))
    return best

def _row_for(rotation, fold, fusion_type, members, weights, strategy, p_cv_fused, y_cv, p_te_fused, y_te, thr):
    if thr is None:
        return {
            'rotation': rotation, 'fold': fold, 'fusion_type': fusion_type,
            'n_members': len(members),
            'members': ','.join(members),
            'weights': ','.join(f'{w:.2f}' for w in weights) if weights else '',
            'strategy': strategy, 'thr': None,
            'cv_f1': np.nan, 'cv_prec': np.nan, 'cv_rec': np.nan,
            'te_f1': np.nan, 'te_prec': np.nan, 'te_rec': np.nan,
            'gap_primary': np.nan, 'cv_n': len(y_cv), 'te_n': len(y_te),
        }
    cv = _metrics_at(p_cv_fused, y_cv, thr)
    te = _metrics_at(p_te_fused, y_te, thr)
    if strategy == 'F1':
        gap = cv['f1'] - te['f1']
    else:
        gap = cv['rec'] - te['rec']
    return {
        'rotation': rotation, 'fold': fold, 'fusion_type': fusion_type,
        'n_members': len(members),
        'members': ','.join(members),
        'weights': ','.join(f'{w:.2f}' for w in weights) if weights else '',
        'strategy': strategy, 'thr': round(float(thr),3),
        'cv_f1': round(cv['f1'],4), 'cv_prec': round(cv['prec'],4), 'cv_rec': round(cv['rec'],4),
        'te_f1': round(te['f1'],4), 'te_prec': round(te['prec'],4), 'te_rec': round(te['rec'],4),
        'gap_primary': round(gap,4), 'cv_n': len(y_cv), 'te_n': len(y_te),
    }

print('Fusion search functions ready.')

In [ ]:
# === 7. Run rotations × folds: fit base models, then search 2-way and 3-way fusions per strategy ===
PAIRS  = list(itertools.combinations(MODELS, 2))
TRIPLES = list(itertools.combinations(MODELS, 3))
all_rows = []
t_total = time.time()

for rot in ROTATIONS:
    rname = rot['name']
    print('\n' + '='*100)
    print(f' ROTATION {rname}   always_train={rot["always"]}   test={rot["test"]}')
    print('='*100)
    for fold_idx in range(N_FOLDS):
        df_tr, df_cv, df_te = build_fold(rot, fold_idx)
        y_cv = df_cv['label_int'].values
        y_te = df_te['label_int'].values
        print(f'\n--- rot {rname} fold {fold_idx} ---  '
              f'tr n={len(df_tr)} (+{int((df_tr["label_int"]==1).sum())})  '
              f'cv n={len(df_cv)} (+{int((y_cv==1).sum())})  '
              f'te n={len(df_te)} (+{int((y_te==1).sum())})')
        # 1) Fit base models, cache probas
        cv_probas, te_probas = {}, {}
        t_fit = time.time()
        for name in MODELS:
            t0 = time.time()
            p_cv, _, p_te, _ = fit_score(name, df_tr, df_cv, df_te, seed=RANDOM_SEED + fold_idx)
            cv_probas[name] = p_cv
            te_probas[name] = p_te
            print(f'    fit {name:20s}  {time.time()-t0:5.1f}s')
        print(f'    [base fits total {time.time()-t_fit:5.1f}s]')

        # 2) Base-model rows (one per model × strategy) — for delta_vs_best_base
        for name in MODELS:
            for s in STRATEGIES:
                if s == 'F1':
                    thr, _ = _best_f1_thr_v(cv_probas[name], y_cv)
                else:
                    thr, _, _ = _best_rec_at_prec_v(cv_probas[name], y_cv, PREC_FLOOR[s])
                all_rows.append(_row_for(rname, fold_idx, 'base', [name], [], s,
                                          cv_probas[name], y_cv, te_probas[name], y_te, thr))

        # 3) 2-way search per strategy
        t_2w = time.time()
        for a, b in PAIRS:
            for s in STRATEGIES:
                hit = search_2way(cv_probas[a], cv_probas[b], y_cv, s)
                if hit is None:
                    all_rows.append(_row_for(rname, fold_idx, '2way', [a,b], [None,None], s,
                                              None, y_cv, None, y_te, None))
                    continue
                alpha, thr, _ = hit
                ws = [alpha, 1-alpha]
                p_cv_f = ws[0]*cv_probas[a] + ws[1]*cv_probas[b]
                p_te_f = ws[0]*te_probas[a] + ws[1]*te_probas[b]
                all_rows.append(_row_for(rname, fold_idx, '2way', [a,b], ws, s,
                                          p_cv_f, y_cv, p_te_f, y_te, thr))
        print(f'    [2-way search {time.time()-t_2w:5.1f}s]')

        # 4) 3-way search per strategy
        t_3w = time.time()
        for a, b, c in TRIPLES:
            for s in STRATEGIES:
                hit = search_3way(cv_probas[a], cv_probas[b], cv_probas[c], y_cv, s)
                if hit is None:
                    all_rows.append(_row_for(rname, fold_idx, '3way', [a,b,c], [None]*3, s,
                                              None, y_cv, None, y_te, None))
                    continue
                ws, thr, _ = hit
                p_cv_f = ws[0]*cv_probas[a] + ws[1]*cv_probas[b] + ws[2]*cv_probas[c]
                p_te_f = ws[0]*te_probas[a] + ws[1]*te_probas[b] + ws[2]*te_probas[c]
                all_rows.append(_row_for(rname, fold_idx, '3way', [a,b,c], list(ws), s,
                                          p_cv_f, y_cv, p_te_f, y_te, thr))
        print(f'    [3-way search {time.time()-t_3w:5.1f}s]')

per_fold = pd.DataFrame(all_rows)
per_fold.to_csv(SAVE_DIR / 'per_fold.csv', index=False)
print(f'\nAll done in {(time.time()-t_total)/60:.1f} min.  Saved per-fold CSV: '
      f'{SAVE_DIR/"per_fold.csv"}  ({len(per_fold)} rows)')

In [ ]:
# === 8. Aggregate across folds + per-(rotation, strategy) display ===
AGG_COLS = ['thr','cv_f1','cv_prec','cv_rec','te_f1','te_prec','te_rec','gap_primary']

def _agg(df):
    means = df[AGG_COLS].mean(numeric_only=True)
    stds  = df[AGG_COLS].std (numeric_only=True)
    out = {f'{c}': round(float(means[c]),4) if not np.isnan(means[c]) else np.nan for c in AGG_COLS}
    for c in ['cv_f1','te_f1','gap_primary']:
        out[f'{c}_std'] = round(float(stds[c]),4) if not np.isnan(stds[c]) else np.nan
    out['n_folds_with_thr'] = int(df['thr'].notna().sum() if df['thr'].dtype != 'O' else len(df))
    return out

summary_rows = []
for (rot, ftype, members, weights_cell, strat), grp in per_fold.groupby(
        ['rotation','fusion_type','members','weights','strategy']):
    summary_rows.append({
        'rotation': rot, 'fusion_type': ftype, 'members': members,
        'weights': weights_cell, 'strategy': strat, **_agg(grp),
    })
summary = pd.DataFrame(summary_rows)
# Per-fusion-candidate avg weights for 2-way/3-way (since weights vary fold-to-fold)
# Replace 'weights' with avg weights across folds for cleaner display.
def _avg_weights(grp):
    if grp['fusion_type'].iloc[0] == 'base':
        return ''
    ws_list = []
    for w in grp['weights']:
        if not w: continue
        ws_list.append([float(x) for x in w.split(',')])
    if not ws_list: return ''
    avg = np.mean(ws_list, axis=0)
    return ','.join(f'{x:.2f}' for x in avg)

agg_w = (per_fold.groupby(['rotation','fusion_type','members','strategy'])
         .apply(_avg_weights).rename('avg_weights').reset_index())
summary_grouped = (per_fold.groupby(['rotation','fusion_type','members','strategy'])
                   .apply(lambda g: pd.Series(_agg(g))).reset_index()
                   .merge(agg_w, on=['rotation','fusion_type','members','strategy'], how='left'))
summary_grouped.to_csv(SAVE_DIR / 'summary_avg.csv', index=False)

def _short_members(s):
    return '+'.join(ALIAS.get(m, m) for m in s.split(','))

for rot in ROTATIONS:
    rname = rot['name']
    print('\n' + '#'*128)
    print(f'#  ROTATION {rname}   always_train={rot["always"]}   test={rot["test"]}')
    print('#'*128)
    for s in STRATEGIES:
        sub = summary_grouped[(summary_grouped['rotation']==rname) & (summary_grouped['strategy']==s)].copy()
        if not len(sub): continue
        primary_col = 'cv_f1' if s == 'F1' else 'cv_rec'
        print('\n' + '='*128)
        if s == 'F1':
            print(f' STRATEGY = {s}  (rank by avg cv_f1; gap = cv_f1 − te_f1)')
        else:
            print(f' STRATEGY = {s}  (CV prec ≥ {int(PREC_FLOOR[s]*100)}%; rank by avg cv_rec; gap = cv_rec − te_rec)')
        print('='*128)
        # Best base + top-K 2way + top-K 3way
        base = sub[sub['fusion_type']=='base'].sort_values(primary_col, ascending=False).head(1)
        twoway = sub[sub['fusion_type']=='2way'].sort_values(primary_col, ascending=False).head(TOP_2WAY)
        threway = sub[sub['fusion_type']=='3way'].sort_values(primary_col, ascending=False).head(TOP_3WAY)
        # Compute delta vs best base for fusions
        if len(base):
            base_score = float(base[primary_col].iloc[0]) if not base[primary_col].isna().all() else np.nan
        else:
            base_score = np.nan
        block = pd.concat([base, twoway, threway], ignore_index=True)
        block['short']           = block['members'].map(_short_members)
        block['delta_cv_vs_base']= (block[primary_col] - base_score).round(4)
        cols = ['fusion_type','short','avg_weights','thr',
                primary_col, primary_col + '_std',
                'te_f1','te_prec','te_rec','gap_primary','delta_cv_vs_base','n_folds_with_thr']
        cols = [c for c in cols if c in block.columns]
        with pd.option_context('display.max_columns', None, 'display.width', 240, 'display.max_colwidth', 60):
            print(block[cols].to_string(index=False, na_rep='  --'))

print(f'\nSaved: {SAVE_DIR/"summary_avg.csv"}  (group by rotation+fusion_type+members+strategy)')
print(f'       {SAVE_DIR/"per_fold.csv"}     (every fold × candidate × strategy row)')

In [ ]:
# === 9. Per-fold detail for the top candidate per (rotation, strategy) ===
# Only the headline winner per (rotation, strategy) is shown — full per-fold detail is in per_fold.csv.
for rot in ROTATIONS:
    rname = rot['name']
    print('\n' + '#'*128)
    print(f'#  ROTATION {rname}  — per-fold breakdown for headline winner per strategy')
    print('#'*128)
    for s in STRATEGIES:
        primary_col = 'cv_f1' if s == 'F1' else 'cv_rec'
        sub_summary = summary_grouped[(summary_grouped['rotation']==rname) &
                                      (summary_grouped['strategy']==s) &
                                      (summary_grouped['fusion_type'].isin(['2way','3way']))]
        if not len(sub_summary): continue
        winner = sub_summary.sort_values(primary_col, ascending=False).iloc[0]
        members = winner['members']; ftype = winner['fusion_type']
        rows = per_fold[(per_fold['rotation']==rname) &
                        (per_fold['strategy']==s) &
                        (per_fold['fusion_type']==ftype) &
                        (per_fold['members']==members)].sort_values('fold')
        cols = ['fold','weights','thr','cv_f1','cv_prec','cv_rec','te_f1','te_prec','te_rec','gap_primary']
        cols = [c for c in cols if c in rows.columns]
        print(f'\n  STRATEGY={s}  winner: {ftype}  {_short_members(members)}')
        with pd.option_context('display.max_columns', None, 'display.width', 200, 'display.max_colwidth', 60):
            print(rows[cols].to_string(index=False, na_rep='  --'))
print('\n(Per-fold detail for non-winning candidates lives in per_fold.csv.)')

## How to read the output

**Cell 8 — headline display.** Two `ROTATION` blocks (A, B), each with one sub-block per strategy. Each sub-block shows:
1. **Top base model** at this strategy (1 row) — the bar to beat.
2. **Top-5 2-way fusions** ranked by avg CV primary metric.
3. **Top-5 3-way fusions** ranked by avg CV primary metric.

Columns:
- `short` — alias-shortened member list (e.g. `t10+wh.xg`, `tst+wv.xg+wh.xg`)
- `avg_weights` — mean weights across folds (per-fold weights are in `per_fold.csv`)
- `thr, cv_*, te_*` — averaged across folds
- `gap_primary` — `cv_primary − te_primary`. **Small symmetric gap on both rotations is the trustworthy signal.**
- `delta_cv_vs_base` — fusion's CV primary − best base's CV primary. Positive ≈ fusion lift on CV. **Look for fusions where `delta_cv_vs_base > 0` AND `gap_primary` stays small** — those are real lifts, not search overfit.
- `cv_f1_std` / `te_f1_std` (or `cv_rec` versions) — fold-to-fold instability. A large std with a large delta is a red flag.

**Cell 9 — per-fold for the headline winner only.** If the winner's `te_f1` swings >0.10 across folds, the fusion is fold-fragile even if the average looks good — prefer a slightly worse but more stable candidate.

**Cross-rotation check.** A fusion that lifts cv_f1 by +0.05 on Rotation A but loses on Rotation B is one-direction-only — it overfits the a4-as-test or a5-as-test direction. Trust fusions that lift symmetrically.

**CSVs** (`checkpoints_fusions/`):
- `per_fold.csv` — every `(rotation, fold, fusion_type, members, weights, strategy)` row. ~6,500 rows.
- `summary_avg.csv` — per `(rotation, fusion_type, members, strategy)` averages + stds + avg_weights. ~1,200 rows.

**Knobs (cell 0):**
- `ALPHA_STEP` — drop to 0.02 for finer 2-way; runtime stays small.
- `SIMPLEX_STEP` — drop to 0.05 (231 points) for finer 3-way; ~3.5x slower.
- `TOP_2WAY` / `TOP_3WAY` — extend the headline display.
- `ROTATIONS` — comment out an entry to run only one direction.

**Why this can be trusted:**
- Test never seen during weight search OR threshold pick. Only used at evaluation.
- Per-strategy weight selection — P95 numbers come from weights optimised for P95, not weights optimised for F1 then evaluated at P95.
- Same protocol as `base_models_cv.ipynb` so the base rows here match those numbers within MC-CV variance — sanity check by spot-comparing one row.